Column mask

In [0]:
%sql
CREATE OR REPLACE FUNCTION claims.governance.mask_beneficiary_id(id STRING)
RETURN CASE
    WHEN is_account_group_member('phi_readers') THEN id
    ELSE CONCAT('XXXX', SUBSTRING(id, -4))
END;

ALTER TABLE claims.silver.beneficiary
  ALTER COLUMN beneficiary_id
  SET MASK claims.governance.mask_beneficiary_id;

In [0]:
%sql
ALTER TABLE claims.silver.beneficiary ALTER COLUMN beneficiary_id DROP MASK;

Row filter

In [0]:
%sql
CREATE OR REPLACE FUNCTION claims.governance.filter_by_state(state STRING)
RETURN is_account_group_member('national_analysts') OR state = '05';

Fallback view

In [0]:
%sql
CREATE OR REPLACE VIEW claims.governance.beneficiary_deidentified AS
SELECT
  SHA2(beneficiary_id, 256) AS beneficiary_hash,
  CASE WHEN age_2010 >= 90 THEN '90+'
       ELSE CAST(FLOOR(age_2010 / 5) * 5 AS STRING) END AS age_band,
  state_code,
  sex_code
FROM claims.silver.beneficiary;

In [0]:
spark.table("claims.silver.beneficiary").select("beneficiary_id").show(3, False)